In [7]:
!pip install -q pandas torch transformers accelerate bitsandbytes tqdm

Python(63292) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


You should consider upgrading via the '/Users/krish/Desktop/SemEval_Task7/SemEval-2026-Task-7/venv/bin/python3 -m pip install --upgrade pip' command.


In [8]:
import pandas as pd
import os

# Define paths based on your folder structure
# Ensure the folder "trial data" is in the same directory as this notebook
base_path = "trial data" 
mcq_path = os.path.join(base_path, "trial_data_multiple_choice.tsv")
saq_path = os.path.join(base_path, "trial_data_unique_answer.tsv")

# Load Data
# We use error_bad_lines=False just in case there are formatting hiccups in the TSV
try:
    df_mcq = pd.read_csv(mcq_path, sep='\t')
    df_saq = pd.read_csv(saq_path, sep='\t')
    print(f"✅ Success! Loaded {len(df_mcq)} MCQ items and {len(df_saq)} SAQ items.")
    display(df_mcq.head(2))
except FileNotFoundError:
    print("❌ Error: Could not find the files. Please ensure your notebook is right next to the 'trial data' folder.")

✅ Success! Loaded 148 MCQ items and 148 SAQ items.


,index,lang_reg,question,multiple_choice_options,correct_answer
0,1,ms-SG,Apakah akronim lazim untuk flat perumahan awam...,DBS\nHPB\nHDB\nSAF,HDB
1,2,ms-SG,Parti politik manakah yang telah menjadi parti...,Parti Pekerja (WP) \nParti Tindakan Rakyat (PA...,Parti Tindakan Rakyat (PAP)


In [9]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# 1. Auto-detect Mac GPU (MPS)
if torch.backends.mps.is_available():
    device = "mps"
    print("✅ Mac GPU (MPS) detected! Using hardware acceleration.")
elif torch.cuda.is_available():
    device = "cuda"
    print("✅ NVIDIA GPU detected!")
else:
    device = "cpu"
    print("⚠️ No GPU detected. Running on CPU (this will be slow).")

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

print(f"Loading {MODEL_ID}...")

# 2. Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# 3. Load Model
# We removed 'load_in_4bit' because it is unstable on Mac without complex setup.
# We use torch.float16 to reduce memory usage by half compared to full precision.
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map=device,
    trust_remote_code=True
)

print("✅ Model loaded successfully.")

/Users/krish/Desktop/SemEval_Task7/SemEval-2026-Task-7/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/krish/Desktop/SemEval_Task7/SemEval-2026-Task-7/venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Mac GPU (MPS) detected! Using hardware acceleration.
Loading Qwen/Qwen2.5-7B-Instruct...


Python(63429) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
`torch_dtype` is deprecated! Use `dtype` instead!
Python(63441) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Fetching 4 files:   0%|                                   | 0/4 [23:37<?, ?it/s]
Cancellation requested; stopping current tasks.


KeyboardInterrupt: 